> ## SOLUTION / ANSWER KEY &mdash; Lab 10.7
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-07-the-eval-set.ipynb`](../lab-07-the-eval-set.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 10.7 &mdash; Build an Eval Set

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Define cases with inputs and expected outputs
- Run an agent over the set and compute a pass-rate
- See why one passing run is not evidence of quality

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`). The **graded** cells assert only on the deterministic parts you build &mdash; guardrail logic, tool wiring, agent structure, and reading a fixed real message trace &mdash; and never call an LLM, so the lab always verifies offline. Cells marked **Optional &mdash; run it for real** call a live local model (`ollama run llama3.2:1b`, or Groq) and self-skip if none is reachable. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; The eval loop](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket
WORK = "/tmp/biaa-lab-10-07"
os.makedirs(WORK, exist_ok=True)
def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening -- the optional live cells self-skip when it isn't."""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False
print("Working dir:", WORK, "| Ollama reachable:", ollama_up())

## Concept
Agent quality is fuzzy and non-deterministic, so *&ldquo;it worked once&rdquo;* is an illusion (deck slide
17). The fix is to **measure**: build an **eval set** &mdash; representative inputs with expected behaviour,
including the failures you've found &mdash; and compute a **pass-rate**. Then you iterate against a target,
not a vibe.

In [ ]:
# A tiny agent under test (deterministic): classifies a query's intent.
def agent_fn(text):
    t = text.lower()
    if "refund" in t: return "billing"
    if "crash" in t: return "tech"
    return "general"
print("agent ready:", agent_fn("I need a refund"))

## Your Turn
Complete `run_eval`: count how many cases the agent gets right and compute the rate.

In [ ]:
def run_eval(fn, cases):
    # cases: list of {input, expected}
    passed = sum(1 for c in cases if fn(c["input"]) == c["expected"])
    return {"passed": passed, "total": len(cases), "rate": passed / len(cases)}

CASES = [
    {"input": "I want a refund", "expected": "billing"},
    {"input": "the app keeps crashing", "expected": "tech"},
    {"input": "what are your hours", "expected": "general"},
]

try:
    print('eval:', run_eval(agent_fn, CASES))
    print('a broken agent:', run_eval(lambda t: 'billing', CASES))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("run_eval counts the passes", lambda: run_eval(agent_fn, CASES)["passed"] == 3)
expect_true("the pass-rate is computed", lambda: run_eval(agent_fn, CASES)["rate"] == 1.0)
expect_true("a perfect agent scores 1.0", lambda: run_eval(agent_fn, CASES)["rate"] == 1.0)
expect_true("a broken agent scores below 1.0", lambda: run_eval(lambda t: "billing", CASES)["rate"] < 1.0)
expect_true("total matches the case count", lambda: run_eval(agent_fn, CASES)["total"] == len(CASES))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
An eval set with a pass-rate turns 'it worked once' into a measurable target you can improve against. It's the engine of the improve step -- and, as the next lab shows, your safety net.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>